# Optional Lab: HTTP Requests, Job Queues, and Workers

**What should an API do when a request takes a long time to finish?**

Imagine clicking **Generate image** or **Create report**. The work takes several seconds,
perhaps minutes. Should the browser keep waiting for the original HTTP response?
Or should the API accept the request now and let the browser collect the result later?

This lab follows that second approach: **submit work → get a job ID → check its status →
collect the result**. Redis holds waiting jobs, and a separate worker processes them.
You do not need to learn Redis commands or its internal data structures for this lab.

**By the end, you should be able to:**
- Explain the roles of the HTTP client, API, queue, and worker.
- Distinguish an accepted request from completed work.
- Follow a job through waiting, processing, success, or failure.
- Extend the existing image lab with Redis and a worker in Docker Compose.

**Suggested timing:** 10 minutes of explanation, 20–25 minutes of live demos, then
30–45 minutes for the Compose assignment. The report demos simulate slow work and
require no OpenAI or Stability credentials.

## 1 · Why put work in a queue?

**Without a queue:** The API receives the request, does the work, and only then returns
the result. This is simple and perfectly reasonable for short operations. For slow work,
the request stays open and may hit a browser, proxy, or server timeout.

**With a queue:** The API records a job and quickly returns its ID. A worker does the work
separately. The client uses another HTTP request to ask whether the result is ready.

A **queue** is a waiting line for work. A **worker** is a separate process that takes jobs
from that line and runs them. **Redis** is a server that holds the queued jobs so the API
and workers can share them. **RQ** is the Python library we use to put jobs into Redis
and run them in workers. Redis itself does not run our report or image-generation code.

![The HTTP API submits jobs; Redis holds the queue; workers produce results](figures/queue-flow.svg)

Why introduce this separation?
- The API can acknowledge a request quickly, even when the work is slow.
- Jobs can wait while workers are busy or temporarily stopped.
- More workers can process more jobs at the same time.

**The trade-off:** There are more services to run, and the client must handle job status
and errors. The queue does not make one image or report faster to generate. If jobs arrive
faster than workers finish them, the waiting line grows.

## 2 · The HTTP conversation

![Submitting a job, receiving 202 and a job ID, then polling for a result](figures/http-job-lifecycle.svg)

1. The client sends **`POST /reports`** with the report request.
2. The API puts a job in the queue and responds **`202 Accepted`** with a `job_id` and
   `status_url`. The `Location` header also points to that status URL.
3. A worker takes the job and processes it.
4. The client periodically sends **`GET /reports/{job_id}`**.
5. When the job succeeds, that response includes the result. If it fails, it includes an error.

**202 means “accepted”, not “finished”.** A successful POST does not guarantee that the
report will be created successfully.

| Job status in this demo | Meaning | What should the client do? |
|---|---|---|
| `queued` | Waiting for a worker | Show a waiting message and check again later |
| `running` | A worker is processing it | Show progress/activity and check again later |
| `succeeded` | The result is ready | Display the result and stop polling |
| `failed` | Processing did not succeed | Show an error and stop polling |

The job status is separate from the HTTP status. In this demo, a successful status lookup
returns **HTTP 200** even when its JSON says `failed`. An unknown job returns **404**;
invalid input returns **422**. If Redis cannot accept a job, the API returns **503**, not 202.

**Ask the class:** If the browser receives 202 and closes, does the accepted job disappear?
No: the worker is a separate process. Closing the browser or stopping polling does not
cancel the job.

## 3 · Set up the live demo

Use **Python 3.12**, Docker, and Linux/macOS or **WSL2 on Windows**. Choose a Jupyter-capable
editor such as VS Code. Run these commands from the repository directory:

```bash
cd optional-lab-redis
python3.12 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
python -m ipykernel install --user --name optional-redis --display-name "Python (Redis Lab)"
```

Open this notebook and choose **Python (Redis Lab)**. If using JupyterLab, install it
with `pip install jupyterlab` and start it with `jupyter lab`.

In a second terminal, start Redis:

```bash
docker run --rm --name optional-redis -p 127.0.0.1:16379:6379 redis:7.4-alpine
```

Leave that terminal running. These demos assume the notebook runs on the same host as
the published Docker port. If your kernel runs inside another container, set `REDIS_URL`
to a Redis address reachable from that container instead.

**For the instructor:** Run the cells in order. The setup helpers launch a worker and API
as separate processes. Their implementation is collapsed so you can focus on the HTTP
flow; expand it if you need to inspect it. The final code cell stops those processes.

In [ ]:
import os
import sys
import time
import uuid
from pathlib import Path

import redis
from rq import Queue
from rq.job import Job

LAB_DIR = Path.cwd()
if (LAB_DIR / "optional-lab-redis" / "redis-introduction.ipynb").exists():
    LAB_DIR = LAB_DIR / "optional-lab-redis"
assert (LAB_DIR / "redis-introduction.ipynb").exists(), "Start in the repository or notebook directory."
os.chdir(LAB_DIR)
RUNTIME = LAB_DIR / ".runtime"
RUNTIME.mkdir(exist_ok=True)

# Use a separate queue for this run, without loading the image lab's API keys.
SESSION = uuid.uuid4().hex[:8]
QUEUE_NAME = f"optional-lab-{SESSION}"
REDIS_URL = os.getenv("REDIS_URL", "redis://127.0.0.1:16379/0")
redis_jobs = redis.Redis.from_url(REDIS_URL)
assert redis_jobs.ping()
queue = Queue(QUEUE_NAME, connection=redis_jobs)
tracked_jobs = []
print("Redis is ready. Demo queue:", QUEUE_NAME)

### Our slow operation

The function below stands in for image generation, exporting a report, or sending an email.
It sleeps briefly so we can observe waiting and processing. Both the API and worker need
access to it, so the cell writes a small shared Python module.

Use the name `FAIL` when you want to demonstrate a processing error.

In [ ]:
%%writefile .runtime/redis_lab_tasks.py
import time


def build_report(name: str, delay: float = 1.0) -> dict:
    # Simulate slow work without calling a paid external service.
    time.sleep(delay)
    if name == "FAIL":
        raise ValueError("Intentional demo error")
    return {"title": name, "message": "Your report is ready."}

In [ ]:
# Instructor setup: start and stop only the processes belonging to this demo.
import atexit
import importlib
import subprocess

if str(RUNTIME) not in sys.path:
    sys.path.insert(0, str(RUNTIME))
importlib.invalidate_caches()
import redis_lab_tasks
importlib.reload(redis_lab_tasks)
processes = {}


def stop_process(name):
    record = processes.pop(name, None)
    if record is None:
        return
    process, log = record
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=8)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    log.close()


def cleanup_processes():
    for name in list(processes):
        stop_process(name)


atexit.register(cleanup_processes)


def start_process(name, arguments):
    assert name not in processes, f"Stop {name} before starting it again."
    env = os.environ.copy()
    env.update(REDIS_URL=REDIS_URL, RQ_QUEUE_NAME=QUEUE_NAME)
    log = (RUNTIME / f"{SESSION}-{name}.log").open("w", encoding="utf-8")
    process = subprocess.Popen(arguments, cwd=RUNTIME, env=env, stdout=log, stderr=log)
    processes[name] = (process, log)
    return process


def start_worker(name="worker"):
    # SimpleWorker suits this local demo. The image lab keeps its normal worker.py.
    return start_process(name, [
        sys.executable, "-c", "from rq.cli import main; main()", "worker",
        "--url", REDIS_URL, "--worker-class", "rq.SimpleWorker",
        "--name", f"optional-lab-{SESSION}-{name}", QUEUE_NAME,
    ])


def wait_for_job(job, timeout=20):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        state = job.get_status(refresh=True)
        if state in {"finished", "failed", "stopped", "canceled"}:
            return state
        time.sleep(0.1)
    raise TimeoutError("Job did not finish. Check the worker log in .runtime/.")

## 4 · A quick look behind the HTTP endpoint

Before adding FastAPI, these few lines show what the API will do: enqueue a function call
and keep the returned job ID. There is no worker running yet.

**Prediction:** Will `enqueue()` return the report, or just a reference to waiting work?

In [ ]:
job = queue.enqueue(redis_lab_tasks.build_report, "First report", 1.0)
tracked_jobs.append(job)
print("Job ID:", job.id)
print("Status before starting a worker:", job.get_status())
assert job.get_status() == "queued"

# Starting the worker is separate from submitting the request.
start_worker()
assert wait_for_job(job) == "finished"
print("The worker produced:", job.return_value())

**Observe:** Submitting a job returns an ID immediately. The result appears only after a
worker processes it. FastAPI will wrap this same operation in an HTTP endpoint.

Now keep the worker running and introduce the client/API conversation.

## 5 · Expose the queue through FastAPI

Our API provides a direct endpoint for comparison, a queued endpoint, and a status endpoint:

| Request | Response |
|---|---|
| `POST /reports/sync` | Waits for the work, then returns the report |
| `POST /reports` | Quickly returns 202, a job ID, and a status URL |
| `GET /reports/{job_id}` | Returns the current job status and any finished result |

The queued endpoint does not start a background thread of its own. It hands the job to
Redis, and the separately running worker takes it from there.

**Instructor focus:** Walk through `enqueue_report` and `report_status`. The rest is setup
and input checking. The normal `def` endpoints allow FastAPI to run these synchronous
calls in its threadpool.

In [ ]:
%%writefile .runtime/redis_lab_api.py
import os

import redis
from fastapi import FastAPI, HTTPException, Response
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field
from rq import Queue
from rq.job import Job
from rq.exceptions import NoSuchJobError

from redis_lab_tasks import build_report

app = FastAPI(title="HTTP Job Queue Demo")
connection = redis.Redis.from_url(os.environ["REDIS_URL"])
queue = Queue(os.environ["RQ_QUEUE_NAME"], connection=connection)


class ReportRequest(BaseModel):
    name: str = Field(min_length=1, max_length=80)
    delay: float = Field(default=1.0, ge=0, le=5)


@app.exception_handler(redis.exceptions.RedisError)
async def redis_unavailable(request, exc):
    # Do not claim acceptance if the queue could not store the job.
    return JSONResponse(status_code=503, content={"detail": "The queue is unavailable."})


@app.post("/reports/sync")
def report_sync(request: ReportRequest):
    # This HTTP request stays open until the report is ready.
    return build_report(request.name, request.delay)


@app.post("/reports", status_code=202)
def enqueue_report(request: ReportRequest, response: Response):
    # Record the work. The worker will execute the function later.
    job = queue.enqueue(build_report, request.name, request.delay)
    status_url = f"/reports/{job.id}"
    response.headers["Location"] = status_url
    return {"job_id": job.id, "status_url": status_url}


@app.get("/reports/{job_id}")
def report_status(job_id: str):
    try:
        job = Job.fetch(job_id, connection=connection)
    except NoSuchJobError:
        raise HTTPException(status_code=404, detail="Job not found or no longer available.")
    if job.origin != queue.name:
        raise HTTPException(status_code=404, detail="Job not found in this demo.")

    # Translate the library's names into our public HTTP API's job states.
    state = job.get_status(refresh=True)
    state = {"started": "running", "finished": "succeeded",
             "stopped": "failed", "canceled": "failed"}.get(state, state)
    body = {"job_id": job.id, "status": state}
    if state == "succeeded":
        body["result"] = job.return_value()
    elif state == "failed":
        body["error"] = "The report could not be created."
    return body

In [ ]:
# Instructor setup: run the API on a free local port and wait until it is ready.
import socket
import httpx

with socket.socket() as sock:
    sock.bind(("127.0.0.1", 0))
    api_port = sock.getsockname()[1]
API_URL = f"http://127.0.0.1:{api_port}"
api_process = start_process("api", [
    sys.executable, "-m", "uvicorn", "redis_lab_api:app",
    "--host", "127.0.0.1", "--port", str(api_port),
])
http = httpx.Client(base_url=API_URL, timeout=10, trust_env=False)
deadline = time.monotonic() + 15
while time.monotonic() < deadline:
    if api_process.poll() is not None:
        raise RuntimeError("The API stopped. Check its .runtime/ log.")
    try:
        if http.get("/openapi.json").status_code == 200:
            break
    except httpx.ConnectError:
        pass
    time.sleep(0.1)
else:
    raise TimeoutError("The API did not become ready.")
print("Open the interactive API documentation:", API_URL + "/docs")

### Demo A · Waiting for the result versus accepting work

Send the same report request to both endpoints. Watch the response time **and the response
body**. A quick response containing an ID is not the report itself.

In [ ]:
payload = {"name": "Monthly report", "delay": 2}

started = time.perf_counter()
direct = http.post("/reports/sync", json=payload)
direct.raise_for_status()
print("Direct request:", direct.status_code, "after", round(time.perf_counter() - started, 2), "seconds")
print(direct.json())

started = time.perf_counter()
accepted = http.post("/reports", json=payload)
assert accepted.status_code == 202
print("Queued request:", accepted.status_code, "after", round(time.perf_counter() - started, 2), "seconds")
print(accepted.json())
print("Location:", accepted.headers["Location"])
assert accepted.headers["Location"] == accepted.json()["status_url"]
tracked_jobs.append(Job.fetch(accepted.json()["job_id"], connection=redis_jobs))

### Demo B · Poll until the work finishes

The client follows the returned status URL. It waits between checks and stops on either
success or failure. A client-side timeout stops our polling loop; it does **not** cancel
an already accepted job.

**Prediction:** Can the first status check already say `succeeded`? Yes, if the worker
finishes before that check. Clients must not assume they will see every intermediate state.

In [ ]:
def poll_report(status_url, timeout=20):
    deadline = time.monotonic() + timeout
    previous = None
    while time.monotonic() < deadline:
        response = http.get(status_url)
        response.raise_for_status()
        body = response.json()
        if body["status"] != previous:
            print("HTTP", response.status_code, "| Job:", body["status"])
            previous = body["status"]
        if body["status"] in {"succeeded", "failed"}:
            return body
        time.sleep(0.3)  # Do not send requests in a tight loop.
    raise TimeoutError("Still waiting. Is a worker running for this queue?")

result = poll_report(accepted.json()["status_url"])
assert result["status"] == "succeeded"
print(result["result"])

### Demo C · The API is available, but the worker is stopped

An available API does not mean a worker is currently available. The API can accept a job
and leave it waiting in Redis. This is useful during a short worker restart, but a long
outage needs monitoring: users would otherwise wait indefinitely.

**Prediction:** Will the POST return an error or 202? What will the status request return?

In [ ]:
stop_process("worker")  # The previous job has finished; do not interrupt active work here.
waiting = http.post("/reports", json={"name": "Report while worker is offline", "delay": 1})
assert waiting.status_code == 202
tracked_jobs.append(Job.fetch(waiting.json()["job_id"], connection=redis_jobs))
status = http.get(waiting.json()["status_url"])
print("POST:", waiting.status_code, "| GET:", status.status_code, status.json())
assert status.json()["status"] == "queued"

# The accepted job is still there. Restart the worker; do not submit it again.
start_worker()
assert poll_report(waiting.json()["status_url"])["status"] == "succeeded"

### Demo D · Accepted work can fail

The POST can succeed while processing fails later. The status endpoint must make that
failure visible so the client can stop its loading indicator instead of polling forever.

In [ ]:
failed = http.post("/reports", json={"name": "FAIL", "delay": 0.5})
assert failed.status_code == 202
tracked_jobs.append(Job.fetch(failed.json()["job_id"], connection=redis_jobs))
print("The API accepted the job:", failed.status_code)
final_status = poll_report(failed.json()["status_url"])
assert final_status["status"] == "failed"
print(final_status)

# These errors happen at the HTTP interface, before successful processing.
assert http.post("/reports", json={"name": ""}).status_code == 422
assert http.get("/reports/unknown-job").status_code == 404
print("Invalid request: 422. Unknown job: 404.")

## 6 · Discuss the mechanics, not the implementation details

- **Waiting order:** Jobs normally wait their turn. With several workers, a short job can
  finish before a longer one that was taken first. Do not promise completion order to clients.
- **Backlog:** If incoming work exceeds worker capacity, jobs wait longer. A quick 202 response
  does not prove the system can finish the work quickly.
- **Retries:** Resending a POST can create a second job. Retrying an accepted job should be
  deliberate, especially for operations such as payments or emails. A job ID identifies
  accepted work; by itself it does not prevent duplicate submissions.
- **Unavailable queue:** If the API cannot put the job in Redis, it should report that failure.
  It must not respond “accepted” and then silently lose the request.
- **Durability:** A worker restart and a Redis restart are different events. Redis holds the
  waiting jobs; keeping them through a Redis/container restart requires persistence setup.

**When is a queue worth it?** Use it when accepting and completing work are genuinely
separate events. Keep a direct response for quick operations. FastAPI `BackgroundTasks`
can run a small task after a response, but that work remains inside the API application;
it does not provide this separate queue-and-worker arrangement.

**Instructor alternative:** Use the displayed `/docs` URL to repeat demos A–D in a browser.
Submit a report, copy its ID, stop/start the worker using the notebook, and query its status.
In a remote environment, forward the displayed port to your browser first.

## 7 · Clean up the live demo

Run this cell after the demonstration, including when you stop before reaching the API
section. It stops only this notebook's processes and removes its recorded jobs.
To repeat the complete demonstration, restart the kernel and run from the top.

In [ ]:
cleanup_processes()
if "http" in globals():
    http.close()
from rq.exceptions import NoSuchJobError
for demo_job in tracked_jobs:
    try:
        demo_job.delete()
    except NoSuchJobError:
        pass
queue.delete(delete_jobs=True)
redis_jobs.close()
print("Demo API and worker stopped; demo jobs removed.")

Stop the dedicated Redis container when you no longer need it:

```bash
docker stop optional-redis
```

The generated demo modules and logs stay in `.runtime/`, which Git ignores.

## 8 · Student Assignment: Add the Queue to Your Compose Lab

Extend your existing [Docker Compose lab](../lab-assignments.ipynb) so that image requests
use Redis and a separate worker. Work in
[`docker-lab/docker-compose.yml`](../docker-lab/docker-compose.yml).

![Frontend, backend, Redis, worker, and shared image storage](figures/compose-architecture.svg)

The existing backend already contains the queued image route. Your task is to connect and
run the services, not to replace it with the report demo:

- `POST /v3/images` accepts work and returns `img_id`.
- `GET /images/{img_id}` returns **202** while the image is missing, then **200** with the JPEG.
- `python worker.py` runs the worker; the queue used by the route is named `default`.
- In Streamlit, select **v3** to test this path.

**Compare the HTTP contracts:** Our report demo returns **202 on POST** and **200 with a JSON
job status on GET**. The existing image lab returns **200 with an ID on POST**, then **202 or
a JPEG on GET**. Both separate acceptance from completion, but the client must follow the
contract of the API it actually calls. Do not rename the existing endpoints for this task.

Relevant code: [routes](../docker-lab/backend/routes/image_gen_router.py),
[queue connection](../docker-lab/backend/dependencies.py),
[worker](../docker-lab/backend/worker.py), and
[frontend](../docker-lab/frontend/streamlit_app.py).

### Build and connect four services

Add **Redis** and a **worker** alongside your existing frontend and backend. The worker
can use the backend image with a different startup command: `python worker.py`.
Use a compatible Python version such as 3.12.

| Service | Configuration you need |
|---|---|
| Frontend | `API_URL=http://backend:8000` |
| Backend and worker | `REDIS_URL=redis://redis:6379/0` |
| Backend and worker | The same `UPLOAD_FOLDER` and a shared image volume |
| Worker | The same task code as the backend; start with `python worker.py` |
| Redis | Readiness check using `redis-cli ping` |

In Compose, services reach each other by **service name**. Inside a container, `localhost`
means that container, so it cannot stand for a different Redis or backend container.
The API and worker should wait until Redis is healthy, not merely started.

The worker writes the JPEG file; the API reads it to serve the HTTP response. That is why
both need the same image volume and a writable `UPLOAD_FOLDER`. Redis holds the job, not
the image file. Make sure the folder exists (`mkdir -p images` for a local backend run).

Provide the image provider credentials through runtime configuration. Unlike the report
demo, the original image lab needs valid API access and credits. Keep keys out of your
screenshots and written submission. The Dockerfiles and Compose file remain your work:
there is no completed solution in this notebook.

### Prove the HTTP flow works

Run these tests from `docker-lab/` and capture the request/response observations:

1. **Start:** `docker compose up --build -d`. Check `docker compose ps` and the worker logs.
2. **Complete a request:** Submit an image request through Streamlit v3 or `POST /v3/images`.
   Keep the `img_id` and poll `GET /images/{img_id}` until it returns the JPEG.
3. **Pause processing:** Run `docker compose stop worker`, then submit another request.
   Explain why the API accepts it but the image is not ready. Start the worker with
   `docker compose start worker` and retrieve the result **without resubmitting the POST**.
4. **Scale processing (optional):** Run `docker compose up -d --scale worker=2`, submit
   several requests, and observe the worker logs. Avoid a fixed `container_name` on a
   service you want to scale. Does the completion order always match submission order?

**Submit:** Your updated Compose file (and any required Dockerfile changes), a small
architecture diagram, and evidence for tests 1–3. Explain the roles of the API, Redis,
and worker; what the POST response means; and how the client knows when to stop waiting.

**HTTP design discussion:** The current image endpoint returns 202 whenever the file is
missing, even for an unknown ID or a failed job. What would a user see? Sketch a better
status response for `queued`, `running`, `succeeded`, and `failed`, and explain how you would
handle an unknown ID. This is a discussion/bonus task, not a required backend rewrite.

**Optional reliability question:** What should happen to accepted jobs if Redis itself
restarts? Explain why stopping a worker is not the same test as removing the queue's data.

## References

- [RQ: submitting jobs and running workers](https://python-rq.org/docs/)
- [FastAPI: background tasks and when separate workers help](https://fastapi.tiangolo.com/tutorial/background-tasks/)
- [HTTP 202 Accepted](https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/202)
- [Docker Compose networking](https://docs.docker.com/compose/how-tos/networking/)
- [Docker Compose startup readiness](https://docs.docker.com/compose/how-tos/startup-order/)

The demo uses the versions in `requirements.txt`, including RQ 1.15.1 to match the image
lab. All diagrams are included in this repository under `figures/`.